# Idea: Combine SetFunTS with convolutional Encoder


Inputs: Triplet (time, value, indicator)

- Reorganize as {channel: (time, value)}
- Group the channels as slow channels and fast channels.
- Apply a convolutional model to the fast channels, and reduce to match slow channels.
    - Alternative: Variable width convolution
- Question: Use 1d convolution? Use 1d convolution with shared params?
- Use 2d convolution over time+value? (or time+value+indicator)?
- Use 2d conv with shared params or 1 per channel?

## Irregular Time Convolution

Convolution: $(f*g)(t) = ∫f(τ)g(t-τ)𝖽τ ≈ ∑_{τᵢ∈𝓝(t)}f(τᵢ)g(t-τᵢ)ω(τᵢ)$

- Pooling: Once a convolutional layer is set up, we ca pool it at arbitrary intermediate points!
    - So where do we actually pool?
    - Pool at observation times of slow channels!
    - Pool at automatically determined times
- Map: $(T⊕ℝ)^* ⟶ (T→ℝᵏ)$


## Convolution with missing values

Simple: Ignore NaN's (only works with 1d convolutions!)





## Implementation Idea 1

- Use separate 2d convolution over time+value (ignore indicator) for each channel


In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

# import logging
# logging.basicConfig(level=logging.INFO)

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.set_printoptions(precision=4, floatmode='fixed', suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7F05586E1D60

In [52]:
from tsdm.tasks import KIWI_FINAL_PRODUCT
task = KIWI_FINAL_PRODUCT()
ts = task.timeseries.sort_index(axis="index").sort_index(axis="columns")
channel_freq = pd.notna(ts).mean().sort_values()

variable
OD600                            0.005413
Fluo_GFP                         0.005468
Glucose                          0.005955
Acetate                          0.006230
Base                             0.020792
Volume                           0.022223
Temperature                      0.394433
DOT                              0.485651
pH                               0.491956
Cumulated_feed_volume_glucose    1.000000
Cumulated_feed_volume_medium     1.000000
Flow_Air                         1.000000
InducerConcentration             1.000000
Probe_Volume                     1.000000
StirringSpeed                    1.000000
dtype: float64

In [53]:
fast_channels = channel_freq[channel_freq >= .1].index
slow_channels = channel_freq[channel_freq <  .1].index

Index(['OD600', 'Fluo_GFP', 'Glucose', 'Acetate', 'Base', 'Volume'], dtype='object', name='variable')

In [54]:
from tsdm.encoders.modular import *

encoder = ChainedEncoder(
    TensorEncoder(names=("time", "value", "index")),
    DataFrameEncoder(
        column_encoders={
            "value": IdentityEncoder(),
            tuple(ts.columns): FloatEncoder("float32"),
        },
        index_encoders=MinMaxScaler() @ DateTimeEncoder(unit="h"),
    ),
    TripletEncoder(sparse=True),
    Standardizer(),
)

ChainedEncoder[TensorEncoder(), DataFrameEncoder(                                                                 col  \
section partition                                                      
index   0                                                       <NA>   
columns 0                                                      value   
        1          [Acetate, Base, Cumulated_feed_volume_glucose,...   

                                           encoder  
section partition                                   
index   0          (MinMaxScaler, DateTimeEncoder)  
columns 0                          IdentityEncoder  
        1                           FloatEncoder()  
), TripletEncoder, Standardizer([-1])]

In [55]:
FAST = ts[fast_channels].dropna(how="all")
SLOW = ts[slow_channels].dropna(how="all")

variable                                   OD600  Fluo_GFP   Glucose  \
run_id experiment_id measurement_time                                  
439    15325         2020-12-09 09:48:38  0.4650    1012.5  4.566546   
                     2020-12-09 11:18:43  1.0850    1042.5  3.894928   
                     2020-12-09 12:48:48  3.0275     872.5  2.745598   
                     2020-12-09 14:18:48  5.0075    1022.5  1.239816   
                     2020-12-09 14:56:35     NaN       NaN       NaN   
...                                          ...       ...       ...   
510    16871         2021-10-26 22:27:45     NaN       NaN       NaN   
                     2021-10-26 22:27:46     NaN       NaN       NaN   
                     2021-10-26 22:30:32     NaN       NaN  0.000000   
                     2021-10-26 22:30:33     NaN       NaN       NaN   
                     2021-10-26 22:41:26     NaN       NaN       NaN   

variable                                   Acetate   Base      Volume  
run_id experiment_id measurement_time                                  
439    15325         2020-12-09 09:48:38  0.115714    NaN         NaN  
                     2020-12-09 11:18:43  0.157560    NaN         NaN  
                     2020-12-09 12:48:48  0.109903    NaN         NaN  
                     2020-12-09 14:18:48  0.063760    NaN         NaN  
                     2020-12-09 14:56:35       NaN   12.0         NaN  
...                                            ...    ...         ...  
510    16871         2021-10-26 22:27:45       NaN  775.0         NaN  
                     2021-10-26 22:27:46       NaN    NaN  -12.692736  
                     2021-10-26 22:30:32  0.049915    NaN         NaN  
                     2021-10-26 22:30:33       NaN    NaN -262.692749  
                     2021-10-26 22:41:26       NaN    NaN -312.692749  

[13049 rows x 6 columns]

In [56]:
ts

variable                                  Acetate  Base  \
run_id experiment_id measurement_time                     
439    15325         2020-12-09 09:10:09      NaN   NaN   
                     2020-12-09 09:10:19      NaN   NaN   
                     2020-12-09 09:10:24      NaN   NaN   
                     2020-12-09 09:10:25      NaN   NaN   
                     2020-12-09 09:10:34      NaN   NaN   
...                                           ...   ...   
510    16871         2021-10-26 22:41:26      NaN   NaN   
                     2021-10-26 22:42:11      NaN   NaN   
                     2021-10-26 22:42:20      NaN   NaN   
                     2021-10-26 22:43:22      NaN   NaN   
                     2021-10-26 22:43:31      NaN   NaN   

variable                                  Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                            0.0   
                     2020-12-09 09:10:19                            0.0   
                     2020-12-09 09:10:24                            0.0   
                     2020-12-09 09:10:25                            0.0   
                     2020-12-09 09:10:34                            0.0   
...                                                                 ...   
510    16871         2021-10-26 22:41:26                         1061.0   
                     2021-10-26 22:42:11                         1061.0   
                     2021-10-26 22:42:20                         1061.0   
                     2021-10-26 22:43:22                         1061.0   
                     2021-10-26 22:43:31                         1061.0   

variable                                  Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                      0.000000   
                     2020-12-09 09:10:19                      0.000000   
                     2020-12-09 09:10:24                      0.000000   
                     2020-12-09 09:10:25                      0.000000   
                     2020-12-09 09:10:34                      0.000000   
...                                                                ...   
510    16871         2021-10-26 22:41:26                   3787.307373   
                     2021-10-26 22:42:11                   3787.307373   
                     2021-10-26 22:42:20                   3787.307373   
                     2021-10-26 22:43:22                   3787.307373   
                     2021-10-26 22:43:31                   3787.307373   

variable                                        DOT  Flow_Air  Fluo_GFP  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09        NaN       0.0       NaN   
                     2020-12-09 09:10:19   0.000000       0.0       NaN   
                     2020-12-09 09:10:24        NaN       0.0       NaN   
                     2020-12-09 09:10:25        NaN       0.0       NaN   
                     2020-12-09 09:10:34   0.000000       0.0       NaN   
...                                             ...       ...       ...   
510    16871         2021-10-26 22:41:26        NaN       5.0       NaN   
                     2021-10-26 22:42:11        NaN       5.0       NaN   
                     2021-10-26 22:42:20  79.459999       5.0       NaN   
                     2021-10-26 22:43:22        NaN       5.0       NaN   
                     2021-10-26 22:43:31  78.860001       5.0       NaN   

variable                                  Glucose  InducerConcentration  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09      NaN                   0.0   
                     2020-12-09 09:10:19      NaN                   0.0   
                 

In [57]:
new = pd.concat((SLOW, FAST), axis="columns")
new = new.sort_index(axis="index").sort_index(axis="columns")

variable                                  Acetate  Base  \
run_id experiment_id measurement_time                     
439    15325         2020-12-09 09:10:09      NaN   NaN   
                     2020-12-09 09:10:19      NaN   NaN   
                     2020-12-09 09:10:24      NaN   NaN   
                     2020-12-09 09:10:25      NaN   NaN   
                     2020-12-09 09:10:34      NaN   NaN   
...                                           ...   ...   
510    16871         2021-10-26 22:41:26      NaN   NaN   
                     2021-10-26 22:42:11      NaN   NaN   
                     2021-10-26 22:42:20      NaN   NaN   
                     2021-10-26 22:43:22      NaN   NaN   
                     2021-10-26 22:43:31      NaN   NaN   

variable                                  Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                            0.0   
                     2020-12-09 09:10:19                            0.0   
                     2020-12-09 09:10:24                            0.0   
                     2020-12-09 09:10:25                            0.0   
                     2020-12-09 09:10:34                            0.0   
...                                                                 ...   
510    16871         2021-10-26 22:41:26                         1061.0   
                     2021-10-26 22:42:11                         1061.0   
                     2021-10-26 22:42:20                         1061.0   
                     2021-10-26 22:43:22                         1061.0   
                     2021-10-26 22:43:31                         1061.0   

variable                                  Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                      0.000000   
                     2020-12-09 09:10:19                      0.000000   
                     2020-12-09 09:10:24                      0.000000   
                     2020-12-09 09:10:25                      0.000000   
                     2020-12-09 09:10:34                      0.000000   
...                                                                ...   
510    16871         2021-10-26 22:41:26                   3787.307373   
                     2021-10-26 22:42:11                   3787.307373   
                     2021-10-26 22:42:20                   3787.307373   
                     2021-10-26 22:43:22                   3787.307373   
                     2021-10-26 22:43:31                   3787.307373   

variable                                        DOT  Flow_Air  Fluo_GFP  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09        NaN       0.0       NaN   
                     2020-12-09 09:10:19   0.000000       0.0       NaN   
                     2020-12-09 09:10:24        NaN       0.0       NaN   
                     2020-12-09 09:10:25        NaN       0.0       NaN   
                     2020-12-09 09:10:34   0.000000       0.0       NaN   
...                                             ...       ...       ...   
510    16871         2021-10-26 22:41:26        NaN       5.0       NaN   
                     2021-10-26 22:42:11        NaN       5.0       NaN   
                     2021-10-26 22:42:20  79.459999       5.0       NaN   
                     2021-10-26 22:43:22        NaN       5.0       NaN   
                     2021-10-26 22:43:31  78.860001       5.0       NaN   

variable                                  Glucose  InducerConcentration  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09      NaN                   0.0   
                     2020-12-09 09:10:19      NaN                   0.0   
                 

In [58]:
pd.testing.assert_frame_equal(ts, new)

In [14]:
encoder = ChainedEncoder(
    # TensorEncoder(names=("time", "value", "index")),
    DataFrameEncoder(
        column_encoders=IdentityEncoder(),
        index_encoders=IdentityEncoder(),
    ),
    Standardizer(),
)

ChainedEncoder[DataFrameEncoder(                    col          encoder
section partition                       
index   0          <NA>  IdentityEncoder
columns 0          <NA>  IdentityEncoder
), Standardizer([-1])]

In [15]:
ts = task.timeseries#.loc[439, 15325]
encoder.fit(ts)
encoded = encoder.encode(ts)
decoded = encoder.decode(encoded)

Flow_Air  StirringSpeed  \
run_id experiment_id measurement_time                               
439    15325         2020-12-09 09:10:09       0.0            0.0   
                     2020-12-09 09:10:19       0.0            0.0   
                     2020-12-09 09:10:24       0.0          100.0   
                     2020-12-09 09:10:25       0.0          100.0   
                     2020-12-09 09:10:34       0.0          100.0   
...                                            ...            ...   
510    16871         2021-10-26 22:41:26       5.0            0.0   
                     2021-10-26 22:42:11       5.0            0.0   
                     2021-10-26 22:42:20       5.0            0.0   
                     2021-10-26 22:43:22       5.0            0.0   
                     2021-10-26 22:43:31       5.0            0.0   

                                          Temperature  Acetate  Base  \
run_id experiment_id measurement_time                                  
439    15325         2020-12-09 09:10:09    36.389999      NaN   NaN   
                     2020-12-09 09:10:19          NaN      NaN   NaN   
                     2020-12-09 09:10:24          NaN      NaN   NaN   
                     2020-12-09 09:10:25    36.130001      NaN   NaN   
                     2020-12-09 09:10:34          NaN      NaN   NaN   
...                                               ...      ...   ...   
510    16871         2021-10-26 22:41:26          NaN      NaN   NaN   
                     2021-10-26 22:42:11    37.450001      NaN   NaN   
                     2021-10-26 22:42:20          NaN      NaN   NaN   
                     2021-10-26 22:43:22    37.439999      NaN   NaN   
                     2021-10-26 22:43:31          NaN      NaN   NaN   

                                          Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                            0.0   
                     2020-12-09 09:10:19                            0.0   
                     2020-12-09 09:10:24                            0.0   
                     2020-12-09 09:10:25                            0.0   
                     2020-12-09 09:10:34                            0.0   
...                                                                 ...   
510    16871         2021-10-26 22:41:26                         1061.0   
                     2021-10-26 22:42:11                         1061.0   
                     2021-10-26 22:42:20                         1061.0   
                     2021-10-26 22:43:22                         1061.0   
                     2021-10-26 22:43:31                         1061.0   

                                          Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                      0.000000   
                     2020-12-09 09:10:19                      0.000000   
                     2020-12-09 09:10:24                      0.000000   
                     2020-12-09 09:10:25                      0.000000   
                     2020-12-09 09:10:34                      0.000000   
...                                                                ...   
510    16871         2021-10-26 22:41:26                   3787.307373   
                     2021-10-26 22:42:11                   3787.307373   
                     2021-10-26 22:42:20                   3787.307373   
                     2021-10-26 22:43:22                   3787.307373   
                     2021-10-26 22:43:31                   3787.307373   

                                                DOT  Glucose  OD600  \
run_id experiment_id measurement_time                                 
439    15325         2020-12-09 09:10:09        NaN      NaN    NaN   
                     2020-12-09 09:10:19   0.000000    